# Tratamento do Dataset de Estudantes — Vértice

Este notebook prepara os dados do conjunto **Student Performance (UCI)** para o contrato oficial de importação **Dados Escolares** do Vértice.

O arquivo final será salvo em `data/estudantes_tratado.csv`, com as 14 colunas do modelo oficial, na ordem esperada e usando `;` como separador.

> **Importante:** o conjunto UCI é um dataset educacional de referência, não um histórico acadêmico real de 2026. O ano, o período e as turmas são valores sintéticos para teste da importação. A distribuição por idade é uma heurística reproduzível, não uma classificação escolar oficial.

## 1. Bibliotecas

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

## 2. Importação dos dados de origem

O arquivo da UCI contém duas bases: Matemática (`student-mat.csv`) e Português (`student-por.csv`). A disciplina é identificada antes da concatenação para que cada registro mantenha sua origem.

In [ ]:
URL_DATASET = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

with urllib.request.urlopen(URL_DATASET, timeout=60) as resposta:
    conteudo_zip = io.BytesIO(resposta.read())

with zipfile.ZipFile(conteudo_zip, "r") as arquivo_principal:
    nomes_arquivos = arquivo_principal.namelist()
    print("Arquivos no pacote UCI:", nomes_arquivos)

    # A distribuição atual da UCI contém os CSVs dentro de student.zip.
    if "student.zip" in nomes_arquivos:
        zip_interno = io.BytesIO(arquivo_principal.read("student.zip"))
        with zipfile.ZipFile(zip_interno, "r") as arquivo_dados:
            df_mat = pd.read_csv(arquivo_dados.open("student-mat.csv"), sep=";")
            df_por = pd.read_csv(arquivo_dados.open("student-por.csv"), sep=";")
    else:
        # Compatibilidade caso os CSVs estejam diretamente no ZIP externo.
        df_mat = pd.read_csv(arquivo_principal.open("student-mat.csv"), sep=";")
        df_por = pd.read_csv(arquivo_principal.open("student-por.csv"), sep=";")

# Identifica a disciplina antes de unir as duas bases.
df_mat["disciplina"] = "MAT"
df_por["disciplina"] = "POR"

print(f"Matemática: {df_mat.shape}")
print(f"Português:  {df_por.shape}")

## 3. Consolidação e auditoria inicial

In [ ]:
df_bruto = pd.concat([df_mat, df_por], ignore_index=True)
df = df_bruto.copy()

print(f"Total de registros: {len(df):,}")
print(f"Registros duplicados considerando todas as colunas: {df.duplicated().sum():,}")
print(f"Total de células nulas: {df.isna().sum().sum():,}")
display(df.head())

## 4. Seleção e nomenclatura das colunas de origem

São mantidos somente os campos necessários ao arquivo escolar e à regra de distribuição das turmas.

- `age` será usado para estimar o ano/série e não será exportado.
- `failures` será renomeado para `historico_reprovacoes`.
- `absences` será renomeado para `faltas`.
- `G1` será convertido em `nota_prova`.
- `G2` será convertido em `nota_trabalho`.
- `G3` não será importado: o arquivo sintético representa somente o **1º Bimestre**, e a regra solicitada usa G1/G2 como componentes de avaliação desse período.

In [ ]:
colunas_origem = ["age", "failures", "absences", "G1", "G2", "disciplina"]
colunas_ausentes = sorted(set(colunas_origem) - set(df.columns))
if colunas_ausentes:
    raise ValueError(f"Colunas esperadas não encontradas na origem: {colunas_ausentes}")

df = df[colunas_origem].copy()

df = df.rename(columns={
    "age": "idade",
    "failures": "historico_reprovacoes",
    "absences": "faltas",
})

# Mantém tipos numéricos consistentes para os cálculos seguintes.
for coluna in ["idade", "historico_reprovacoes", "faltas", "G1", "G2"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="raise")

print("Colunas após seleção e renomeação:")
print(df.columns.tolist())

## 5. Identificador anônimo

O notebook já cria a coluna técnica `id_aluno`. Ela será **renomeada** para `codigo_anonimo`, sem criar um segundo identificador. O padrão `ALU_0001` mantém o código alfanumérico e não utiliza nome ou dado civil do estudante.

Como a base UCI não fornece um identificador individual compartilhado entre as duas disciplinas, os códigos seguem a unidade de registro de origem (cada linha/observação de disciplina).

In [ ]:
df.insert(0, "id_aluno", [f"ALU_{i + 1:04d}" for i in range(len(df))])
df = df.rename(columns={"id_aluno": "codigo_anonimo"})

assert df["codigo_anonimo"].is_unique, "O código anônimo deve ser único por registro preparado."
assert df["codigo_anonimo"].str.match(r"^ALU_\d{4,}$").all()

## 6. Conversão das notas e cálculo de `nota`

Na base UCI, G1 e G2 estão na escala de 0 a 20. Para adequá-las à escala de 0 a 10 utilizada pelo Vértice, dividimos os valores por 2. Em seguida:

- `nota_prova` recebe G1 convertido;
- `nota_trabalho` recebe G2 convertido;
- `nota` é a média aritmética entre `nota_prova` e `nota_trabalho`.

G3 não participa desses cálculos.

In [ ]:
df["nota_prova"] = (df.pop("G1") / 2).round(1)
df["nota_trabalho"] = (df.pop("G2") / 2).round(1)
df["nota"] = ((df["nota_prova"] + df["nota_trabalho"]) / 2).round(1)

assert df[["nota_prova", "nota_trabalho", "nota"]].ge(0).all().all()
assert df[["nota_prova", "nota_trabalho", "nota"]].le(10).all().all()

display(df[["codigo_anonimo", "disciplina", "nota_prova", "nota_trabalho", "nota"]].head())

## 7. Inclusão sintética do ano e do período letivo

Estas duas colunas **não existem na base UCI**. São acrescentadas artificialmente apenas para adequação ao modelo de importação e para os testes do sistema:

- `ano_letivo`: `2026`
- `periodo_letivo`: `1º Bimestre`

Atenção: o uso desses valores não significa que os registros tenham sido coletados em uma escola brasileira no ano de 2026.

In [ ]:
ANO_LETIVO = 2026
PERIODO_LETIVO = "1º Bimestre"

df["ano_letivo"] = ANO_LETIVO
df["periodo_letivo"] = PERIODO_LETIVO

## 8. Distribuição sintética das turmas com base na idade

A idade é uma **referência inicial**, não uma regra absoluta. A distribuição adotada é:

| Idade | Distribuição sugerida |
|---|---|
| 15 anos | 1º ano |
| 16 anos | Divisão entre 1º e 2º ano |
| 17 anos | Divisão entre 2º e 3º ano |
| 18 anos ou mais | 3º ano, considerando distorção idade-série |

Dentro de cada ano, os registros são distribuídos entre as turmas A, B, C, D e E. A alocação é balanceada e utiliza uma semente fixa (`42`), portanto o resultado é reproduzível.

Os nomes seguem o padrão documentado no projeto (`Turma 1A`, `Turma 2A` etc.). **Antes de importar**, cadastre no Vértice as 15 turmas com esses nomes, `ano_letivo=2026` e as séries correspondentes (`1º Ano`, `2º Ano` e `3º Ano`).

In [ ]:
SEMENTE_ALEATORIA = 42
LETRAS_TURMA = np.array(list("ABCDE"))
rng = np.random.default_rng(SEMENTE_ALEATORIA)

# A série é atribuída por idade com divisão equilibrada para 16 e 17 anos.
serie_por_indice = pd.Series(index=df.index, dtype="int64")
serie_por_indice.loc[df["idade"] <= 15] = 1
serie_por_indice.loc[df["idade"] >= 18] = 3

# Idades de transição: metade (ou diferença máxima de um registro) para cada série adjacente.
def dividir_idade_entre_series(indices, serie_a, serie_b, gerador):
    indices_embaralhados = gerador.permutation(np.asarray(list(indices), dtype=int))
    serie_por_indice.loc[indices_embaralhados[::2]] = serie_a
    serie_por_indice.loc[indices_embaralhados[1::2]] = serie_b

indices_16 = df.index[df["idade"] == 16]
indices_17 = df.index[df["idade"] == 17]
dividir_idade_entre_series(indices_16, 1, 2, rng)
dividir_idade_entre_series(indices_17, 2, 3, rng)

if serie_por_indice.isna().any():
    idades_sem_regra = sorted(df.loc[serie_por_indice.isna(), "idade"].unique().tolist())
    raise ValueError(f"Há idades sem regra de distribuição definida: {idades_sem_regra}")

df["serie_ano"] = serie_por_indice.astype(int)

# Distribui os registros de cada série de forma cíclica entre A-E, após embaralhamento.
df["turma"] = ""
for serie in (1, 2, 3):
    indices_serie = df.index[df["serie_ano"] == serie].to_numpy()
    indices_serie = rng.permutation(indices_serie)
    for posicao, indice in enumerate(indices_serie):
        letra = LETRAS_TURMA[posicao % len(LETRAS_TURMA)]
        df.loc[indice, "turma"] = f"Turma {serie}{letra}"

mapa_series = {1: "1º Ano", 2: "2º Ano", 3: "3º Ano"}
df["serie"] = df["serie_ano"].map(mapa_series)

print("Distribuição por série:")
display(df.groupby(["serie", "idade"], observed=True).size().unstack(fill_value=0))
print("Distribuição por turma:")
display(df["turma"].value_counts().sort_index())

## 9. Frequência e faltas

`faltas` recebe o valor original de `absences`. Mantém-se a premissa de 200 dias letivos usada no tratamento anterior para calcular `frequencia_percentual`:

\[\text{frequência} = \frac{200 - \text{faltas}}{200} \times 100\]

Essa é uma estimativa anual baseada na premissa do notebook original. Como a linha preparada está identificada como 1º Bimestre, a frequência não deve ser interpretada como uma apuração real apenas desse bimestre.

In [ ]:
DIAS_LETIVOS_ANO_REFERENCIA = 200

df["frequencia_percentual"] = (
    (DIAS_LETIVOS_ANO_REFERENCIA - df["faltas"]) / DIAS_LETIVOS_ANO_REFERENCIA * 100
).clip(0, 100).round(1)

assert df["faltas"].ge(0).all()
assert df["frequencia_percentual"].between(0, 100).all()

## 10. Média do período e média geral

`nota` representa a média das duas avaliações (prova e trabalho) no período. `media_geral` é calculada como a média das notas (`nota`) disponíveis para o mesmo código anônimo, disciplina e ano letivo.

Neste arquivo há apenas um período (`1º Bimestre`); portanto, a média geral inicial coincide com `nota`. **Limitação importante:** o código anônimo sintético (`ALU_0001`, etc.) é criado por observação nesta preparação. Para calcular uma média anual real entre diferentes importações/períodos, o mesmo `codigo_anonimo` deverá ser preservado para o mesmo estudante e disciplina em todos os períodos. Esta base de referência não fornece uma chave que permita relacionar com segurança registros de períodos futuros.

In [ ]:
chaves_media_geral = ["codigo_anonimo", "disciplina", "ano_letivo"]
df["media_geral"] = (
    df.groupby(chaves_media_geral)["nota"]
      .transform("mean")
      .round(1)
)

# O dataset de origem não informa se houve participação em recuperação nem um
# desfecho final escolar. Campos opcionais permanecem vazios (desconhecidos/em andamento).
df["participacao_recuperacao"] = ""
df["situacao_final"] = ""

## 11. Montagem do arquivo no contrato oficial do Vértice

A sequência abaixo reproduz exatamente as 14 colunas de `modelo_dados_escolares.csv`. Campos auxiliares como `idade` e `serie` são utilizados no tratamento, mas não fazem parte do arquivo final.

In [ ]:
COLUNAS_MODELO_DADOS_ESCOLARES = [
    "codigo_anonimo",
    "turma",
    "disciplina",
    "ano_letivo",
    "periodo_letivo",
    "frequencia_percentual",
    "faltas",
    "media_geral",
    "nota",
    "nota_prova",
    "nota_trabalho",
    "historico_reprovacoes",
    "participacao_recuperacao",
    "situacao_final",
]

df_importacao = df[COLUNAS_MODELO_DADOS_ESCOLARES].copy()

print("Colunas prontas para importação:")
print(df_importacao.columns.tolist())
print(f"Quantidade de registros: {len(df_importacao):,}")
display(df_importacao.head(10))

## 12. Validações finais

As verificações abaixo impedem que o notebook exporte um arquivo com cabeçalho divergente, dados obrigatórios vazios ou valores fora das faixas básicas do contrato.

In [ ]:
# 1) Contrato exato de colunas e quantidade de registros.
assert df_importacao.columns.tolist() == COLUNAS_MODELO_DADOS_ESCOLARES
assert len(df_importacao) == len(df_bruto)

# 2) Campos estruturais e identificadores.
campos_estruturais = ["codigo_anonimo", "turma", "disciplina", "ano_letivo", "periodo_letivo"]
assert df_importacao[campos_estruturais].notna().all().all()
assert df_importacao["codigo_anonimo"].astype(str).str.strip().ne("").all()
assert df_importacao["codigo_anonimo"].is_unique
assert set(df_importacao["ano_letivo"].unique()) == {ANO_LETIVO}
assert set(df_importacao["periodo_letivo"].unique()) == {PERIODO_LETIVO}
assert set(df_importacao["disciplina"].unique()).issubset({"MAT", "POR"})

# 3) Turmas dentro do conjunto previsto.
turmas_previstas = {f"Turma {serie}{letra}" for serie in (1, 2, 3) for letra in "ABCDE"}
assert set(df_importacao["turma"].unique()).issubset(turmas_previstas)

# 4) Faixas numéricas e consistência entre notas.
for coluna in ["nota", "nota_prova", "nota_trabalho", "media_geral"]:
    assert df_importacao[coluna].between(0, 10).all(), f"Valores inválidos em {coluna}"
assert df_importacao["frequencia_percentual"].between(0, 100).all()
assert df_importacao["faltas"].ge(0).all()
assert df_importacao["historico_reprovacoes"].between(0, 50).all()

# Nota = média de prova e trabalho.
media_componentes = ((df_importacao["nota_prova"] + df_importacao["nota_trabalho"]) / 2).round(1)
assert media_componentes.equals(df_importacao["nota"].reset_index(drop=True))

# Recuperação e situação final ficam vazias intencionalmente, não são zero/Não inferido.
assert df_importacao["participacao_recuperacao"].eq("").all()
assert df_importacao["situacao_final"].eq("").all()

print("Todas as validações estruturais e numéricas foram aprovadas.")

## 13. Análise exploratória resumida dos dados preparados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="serie", order=["1º Ano", "2º Ano", "3º Ano"], ax=ax)
ax.set_title("Registros distribuídos por série")
ax.set_xlabel("Série atribuída pela regra sintética de idade")
ax.set_ylabel("Quantidade de registros")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df_importacao, x="nota", hue="disciplina", bins=20, kde=True, ax=ax)
ax.set_title("Distribuição das notas médias no período")
ax.set_xlabel("Nota (escala de 0 a 10)")
ax.set_ylabel("Quantidade de registros")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ordem_turmas = sorted(df_importacao["turma"].unique())
sns.countplot(data=df_importacao, x="turma", order=ordem_turmas, ax=ax)
ax.set_title("Quantidade de registros por turma sintética")
ax.set_xlabel("Turma")
ax.set_ylabel("Quantidade de registros")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 14. Exportação do CSV compatível com o modelo

O importador oficial do Vértice detecta o separador do CSV automaticamente. Ainda assim, a exportação usa `;`, igual ao modelo anexado, e codificação `UTF-8-SIG` para preservar acentos ao abrir o arquivo em ferramentas comuns.

In [ ]:
# Funciona tanto quando o notebook é executado na raiz do repositório
# quanto quando é executado dentro da pasta notebooks/.
candidatos_data = [Path.cwd() / "data", Path.cwd().parent / "data"]
pasta_data = next((p for p in candidatos_data if p.is_dir()), Path.cwd() / "data")
pasta_data.mkdir(parents=True, exist_ok=True)

CAMINHO_SAIDA = pasta_data / "estudantes_tratado.csv"
df_importacao.to_csv(
    CAMINHO_SAIDA,
    index=False,
    sep=";",
    encoding="utf-8-sig",
)

print(f"Arquivo exportado: {CAMINHO_SAIDA.resolve()}")
print(f"Tamanho: {CAMINHO_SAIDA.stat().st_size:,} bytes")

## 15. Conferência do arquivo exportado

In [ ]:
# Reabre o CSV com o mesmo padrão do modelo e confere o cabeçalho.
df_conferido = pd.read_csv(
    CAMINHO_SAIDA,
    sep=";",
    encoding="utf-8-sig",
    keep_default_na=False,
)

assert df_conferido.columns.tolist() == COLUNAS_MODELO_DADOS_ESCOLARES
assert len(df_conferido) == len(df_importacao)

print("Cabeçalho conferido:")
print(";".join(df_conferido.columns))
print(f"Linhas exportadas: {len(df_conferido):,}")
display(df_conferido.head())

## Observações para a importação no Vértice

1. Cadastre previamente as turmas com os nomes exatos `Turma 1A` a `Turma 1E`, `Turma 2A` a `Turma 2E` e `Turma 3A` a `Turma 3E`. Todas devem estar vinculadas ao ano letivo de 2026 e às séries `1º Ano`, `2º Ano` ou `3º Ano`.
2. Confirme que as disciplinas `MAT` e `POR` existem na escola e que o usuário que fará a importação possui vínculo/autorização para as turmas e disciplinas correspondentes.
3. `participacao_recuperacao` e `situacao_final` são exportadas vazias intencionalmente: a fonte não contém essas informações para o registro do primeiro bimestre.
4. A frequência foi estimada com a premissa de 200 dias letivos anuais, herdada do tratamento anterior. Ela não representa uma frequência bimestral observada.
5. `media_geral` coincide com `nota` neste arquivo porque existe somente o 1º Bimestre. Ela deve ser atualizada à medida que outros períodos do mesmo estudante/disciplina forem registrados.
6. A distribuição de turma por idade é sintética para teste. Para uso escolar real, a alocação deve ser conferida e ajustada pela equipe pedagógica.